<h1> Project 1- Build a Database </h1>
<p> This Jupyter notebook contains the code that I used to build the database and store the data that I will be using in the analysis. Although this was a relatively smaller project in which the deployment of a database was not necessary, I decided to do so because of the two characteristics of a relational database. First, a database is updatable, and second, tables of a database are independent. These two characteristics of a database make the analysis of this project updatable and carried out with different purposes as long as new data are stored. In what follows, I will first present a database diagram that I made with dbdiagram.io to visualize the structure, and then I will move directly to the codes where I build the database, which includes one part where I ran a web scraper to collect the data I want and stored to the database. The SQL I used in this project was MySQL.</p>

<h2> Database Diagram </h2>

<img src="kpopcomment.png" width=600 height=800 />

<h2> DROP, CREATE and INSERT with MySQL </h2>

In [ ]:
from mysql.connector import connect, Error

import pandas as pd
import datetime as dt
import re

In [2]:
### connection parameters

h = #host
u = #user
pw = #password

In [3]:
#
# create database who_can_be_kpop
#

In [4]:
#drop_database_query = """

#    DROP DATABASE who_can_be_kpop
    
#    """

#try:
#    with connect(host=h, user=u, password=pw) as connection:
        
#        with connection.cursor() as cursor:            
#            cursor.execute(drop_database_query)
        
#except Error as e:
#    print(e)

In [5]:
create_database_query = """

    CREATE DATABASE who_can_be_kpop
    
    """

try:
    with connect(host=h, user=u, password=pw) as connection:
        
        with connection.cursor() as cursor:            
            cursor.execute(create_database_query)
        
except Error as e:
    print(e)

In [6]:
def wcbk_table_execute(query):

    try:
        with connect(host=h, user=u, password=pw, database='who_can_be_kpop') as connection:
        
            with connection.cursor() as cursor:            
                cursor.execute(query)
                connection.commit()
    except Error as e:
        print(e)

#
# create table group_list, idol_info, video_list
#

create_table_group_list = """

    CREATE TABLE group_list(
        id INT AUTO_INCREMENT PRIMARY KEY,
        group_name VARCHAR(100),
        debuted_date DATE,
        debuted_under VARCHAR(100)
        )
        """

create_table_idol_info = """

    CREATE TABLE idol_info(
        id INT AUTO_INCREMENT PRIMARY KEY,
        name VARCHAR(30),
        group_id INT,
        group_name VARCHAR(30),
        nationality VARCHAR(30),
        ethnicity1 VARCHAR(30),
        ethnicity2 VARCHAR(30),
        ethnicity3 VARCHAR(30),
        active BOOLEAN,
        
        FOREIGN KEY(group_id) REFERENCES group_list(id)
        )
        """

create_table_video_list = """

    CREATE TABLE video_list(
        id INT AUTO_INCREMENT PRIMARY KEY,
        video_title VARCHAR(100),
        song_name VARCHAR(100),
        group_id INT,
        released_date DATE,
        youtube_add VARCHAR(1000),
        
        FOREIGN KEY(group_id) REFERENCES group_list(id)
        ) DEFAULT CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci
        """

wcbk_table_execute(create_table_group_list)
wcbk_table_execute(create_table_idol_info)
wcbk_table_execute(create_table_video_list)

In [7]:
#
# insert data to table group_list
#

In [8]:
insert_group_list_query = """

    INSERT INTO group_list(group_name, debuted_date, debuted_under)
        VALUES ( %s, %s, %s)
        
        """
group_records = [
    
    ('blackswan', dt.date(2020, 10, 16), 'dr music'),
    ('prisma', dt.date(2020, 10, 31), 'unionwave entertainment'),
    ('uhsn', None, None),
    ('exp edition', dt.date(2017, 4, 17), 'immabb entertainment'),
    ('sb19', dt.date(2018, 10, 16), 'showbt philippines'),
    ('kaachi', dt.date(2020, 4, 15), 'frontrow records'),
    ('5 high', None, 'livon and 9xo'),
    ('cherry bullet', dt.date(2019, 1, 21), 'fnc entertainment'),
    ('fanatics', dt.date(2019, 8, 6), 'color star technology'),
    ('woo!ah!', dt.date(2020, 5, 15), 'nv entertainment')
    
    ]
try:
    with connect(host=h, user=u, password=pw, database='who_can_be_kpop') as connection:
        
        with connection.cursor() as cursor:            
                cursor.executemany(insert_group_list_query, group_records)
                connection.commit()
except Error as e:
    print(e)

In [9]:
#
# insert data to table group_list
#

In [10]:
def find_group_id(group_name):
    
    #
    #  group name: string, lowercase only, space allowed
    #
    
    paired_group_id = dict()

    select_paired_group_id = """ SELECT id, group_name FROM group_list """

    try:
        with connect(host=h, user=u, password=pw, database='who_can_be_kpop') as connection:
        
            with connection.cursor() as cursor:            
                cursor.execute(select_paired_group_id)
                for row in cursor.fetchall():
                
                    a, b = row
                    paired_group_id[b] = a
                
                
    except Error as e:
        print(e)
    
    return(paired_group_id[group_name])

In [11]:
insert_idol_info_query = """

    INSERT INTO idol_info(name, group_id, group_name, nationality,
        ethnicity1, ethnicity2, ethnicity3, active)
        VALUES( %s, %s, %s, %s, %s, %s, %s, %s)
        
        """

idol_records = [
    
    ('youngheun', find_group_id('blackswan'), 'blackswan', 'korean', 'korean', None, None, 1),
    ('fatou', find_group_id('blackswan'), 'blackswan', 'senegalese', 'senegalese', None, None, 1),
    ('judy', find_group_id('blackswan'), 'blackswan', 'korean', 'korean', None, None, 1),
    ('Leia', find_group_id('blackswan'), 'blackswan', 'brazilian', 'brazilian', 'japanese', None, 1),
    ('hyeme', find_group_id('blackswan'), 'blackswan', 'korean', 'korean', None, None, 0),
    
    ('gyeongmin', find_group_id('prisma'), 'prisma', 'korean', 'korean', None, None, 1),
    ('eunbyeol', find_group_id('prisma'), 'prisma', 'korean', 'korean', None, None, 1),
    ('sally', find_group_id('prisma'), 'prisma', 'american', 'chinese', None, None, 1),
    ('nia', find_group_id('prisma'), 'prisma', 'spainish', 'spainish', None, None, 1),
    ('mariam', find_group_id('prisma'), 'prisma', 'italian', 'italian', None, None, 1),
    
    ('deesee', find_group_id('uhsn'), 'uhsn', 'russian', 'russian', None, None, 1),
    ('livia', find_group_id('uhsn'), 'uhsn', 'swedish', 'swedish', None, None, 1),
    ('mind', find_group_id('uhsn'), 'uhsn', 'thai', 'thai', None, None, 1),
    ('nada', find_group_id('uhsn'), 'uhsn', 'egyptian', 'egyptian', None, None, 1),
    ('oline', find_group_id('uhsn'), 'uhsn', 'norwegian', 'norwegian', None, None, 1),
    ('maria', find_group_id('uhsn'), 'uhsn', 'american', 'american', None, None, 1),
    ('vlada', find_group_id('uhsn'), 'uhsn', 'ukrainian', 'ukrainian', None, None, 1),
    ('luna', find_group_id('uhsn'), 'uhsn', 'polish', 'polish', None, None, 1),
    ('liisu', find_group_id('uhsn'), 'uhsn', 'estonian', 'estonian', None, None, 1),
    ('erii', find_group_id('uhsn'), 'uhsn', 'japanese', 'japanese', None, None, 1),
    
    ('frankie', find_group_id('exp edition'), 'exp edition', 'american', 'american', None, None, 1),
    ('hunter', find_group_id('exp edition'), 'exp edition', 'american', 'american', None, None, 1),
    ('sime', find_group_id('exp edition'), 'exp edition', 'american', 'croatian', None, None, 1),
    ('koki', find_group_id('exp edition'), 'exp edition', None, 'german', 'japanese', None, 1),
    
    ('pablo', find_group_id('sb19'), 'sb19', 'filipino', 'filipino', None, None, 1),
    ('josh', find_group_id('sb19'), 'sb19', 'filipino', 'filipino', None, None, 1),
    ('stell', find_group_id('sb19'), 'sb19', 'filipino', 'filipino', None, None, 1),
    ('ken', find_group_id('sb19'), 'sb19', 'filipino', 'filipino', None, None, 1),
    ('justin', find_group_id('sb19'), 'sb19', 'filipino', 'filipino', None, None, 1),
    
    ('nicole', find_group_id('kaachi'), 'kaachi', None, 'venezuelan', None, None, 1),
    ('coco', find_group_id('kaachi'), 'kaachi', 'korean', 'korean', None, None, 1),
    ('chunseo', find_group_id('kaachi'), 'kaachi', 'spain', 'spain', 'filipino', None, 1),
    ('dani', find_group_id('kaachi'), 'kaachi', 'uk', 'uk', 'irish', None, 0),
    
    ('hitali vernekar', find_group_id('5 high'), '5 high', 'indian', 'indian', None, None, 1),
    ('nana secii', find_group_id('5 high'), '5 high', 'indian', 'indian', None, None, 1),
    ('dhevi hangu pombu', find_group_id('5 high'), '5 high', 'indian', 'indian', None, None, 1),
    ('nasa rianchi', find_group_id('5 high'), '5 high', 'indian', 'indian', None, None, 1),
    ('maria mize', find_group_id('5 high'), '5 high', 'indian', 'indian', None, None, 1),
    
    ('haeyoon', find_group_id('cherry bullet'), 'cherry bullet', 'korean', 'korean', None, None, 1),
    ('yuju', find_group_id('cherry bullet'), 'cherry bullet', 'korean', 'korean', None, None, 1),
    ('bora', find_group_id('cherry bullet'), 'cherry bullet', 'korean', 'korean', None, None, 1), 
    ('jiwon', find_group_id('cherry bullet'), 'cherry bullet', 'korean', 'korean', None, None, 1),
    ('remi', find_group_id('cherry bullet'), 'cherry bullet', 'japanese', 'japanese', None, None, 1),
    ('chaerin', find_group_id('cherry bullet'), 'cherry bullet', 'korean', 'korean', None, None, 1),
    ('may', find_group_id('cherry bullet'), 'cherry bullet', 'japanese', 'japanese', None, None, 1),
    ('mirae', find_group_id('cherry bullet'), 'cherry bullet', 'korean', 'korean', None, None, 0),
    ('kokoro', find_group_id('cherry bullet'), 'cherry bullet', 'japanese', 'japanese', None, None, 0),
    ('linlin', find_group_id('cherry bullet'), 'cherry bullet', 'taiwanese', 'taiwanese', None, None, 0),
    
    ('doi', find_group_id('fanatics'), 'fanatics', 'korean', 'korean', None, None, 1),
    ('sika', find_group_id('fanatics'), 'fanatics', 'chinese', 'chinese', 'japanese', None, 1),
    ('chaelin', find_group_id('fanatics'), 'fanatics', 'korean', 'korean', None, None, 1),
    ('chiayi', find_group_id('fanatics'), 'fanatics', 'taiwanese', 'taiwanese', None, None, 1),
    ('via', find_group_id('fanatics'), 'fanatics', 'korean', 'korean', None, None, 1),
    ('yoonhye', find_group_id('fanatics'), 'fanatics', 'korean', 'korean', None, None, 1),
    ('rayeon', find_group_id('fanatics'), 'fanatics', 'korean', 'korean', None, None, 1),
    
    ('nana', find_group_id('woo!ah!'), 'woo!ah!', 'korean', 'korean', None, None, 1),
    ('wooyeon', find_group_id('woo!ah!'), 'woo!ah!', 'korean', 'korean', None, None, 1),
    ('sora', find_group_id('woo!ah!'), 'woo!ah!', 'japanese', 'japanese', None, None, 1),
    ('lucy', find_group_id('woo!ah!'), 'woo!ah!', 'korean', 'korean', None, None, 1),
    ('minseo', find_group_id('woo!ah!'), 'woo!ah!', 'korean', 'korean', None, None, 1),
    ('songyee', find_group_id('woo!ah!'), 'woo!ah!', 'korean', 'korean', None, None, 0)
    
    ]

try:
    with connect(host=h, user=u, password=pw, database='who_can_be_kpop') as connection:
        
        with connection.cursor() as cursor:            
                cursor.executemany(insert_idol_info_query, idol_records)
                connection.commit()
except Error as e:
    print(e)

<h2> Scraping Comments from YouTube Videos and Storing Them into Database </h2>

In [13]:
import time
from selenium.webdriver import Chrome
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [14]:
#
#    enter your driver path
#

driver_path = r'C:\Users\chunj\webdriver\chromedriver.exe'

#
#    defining a function that can insert a data in table video_list
#                      ,make a new table that store the comment data.
#                   and link the comment table to the table video_list
#

def scrapper(data_dict, numbers_of_comments):
    
    #####################################################
    #
    #    data_dict is a data with value as a tuple that
    #               consist of three items (group_id, song_name, video_address)
    #    numbers_of_comments is an integer
    #
    #####################################################
        
    #
    # starting with finding all information needed to insert a data in video_list
    #
    
    for i, n, a in data_dict.values():
        
        group_id = i
        song_name = n
        v_address = a
        
    with Chrome(executable_path=driver_path) as driver:
        wait = WebDriverWait(driver,15)
        driver.get(v_address)
            
        #scroll down a little for comment section to show, otherwise the program cannot be initiated
        driver.execute_script("window.scrollTo(0, 400)")
        time.sleep(2)
        driver.execute_script("window.scrollTo(0, 400)")
        time.sleep(2)
        driver.execute_script("window.scrollTo(0, 400)")
        time.sleep(2)
            
        #
        # scrap basic information of the video including video_title ande released_date,
        # which will be stored into database
        #
                
        # scrap video_title
                
        for title in wait.until(EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "#container > h1 > yt-formatted-string"))):
                    
            video_title = title.text
                    
        # scrap released_date
                    
        for r_date in wait.until(EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "#info-strings > yt-formatted-string"))):
                    
            r_date = r_date.text
        
        #####################################
        #   output 1: row to insert later
        #####################################
        
        row = (video_title, song_name, group_id, r_date, v_address)
            
        #
        # scraping comments process begins,
        # starting by scroll down the times that are enough to show the amount of comments that
        # is planned to be scraped.
        #
    
        table = pd.DataFrame()
        
        table['id'] = 0
        
        comment_data = []
        author_data = []
            
        Except = 0
    
        #
        # scrolling 1 time show approximately 20 comments
        #
    
        scroll_times = int(numbers_of_comments/20)
                
        for s in range(scroll_times): 
            wait.until(EC.visibility_of_element_located((By.TAG_NAME, "body"))).send_keys(Keys.END)
            time.sleep(10)
                
        #
        #  the code below try to click all link with tag more, 
        #  so longer comments can be scrapped fully as well.
        #  The code will pass if it finds something cannot click and raise an Exception
        #
                
        for link in driver.find_elements_by_css_selector('#more'):
            try:
                driver.execute_script("arguments[0].click();", link)
            except Exception:
                Except += 1
                pass
                    
        time.sleep(10)
        
        #
        #  one significant drawback of this scrapper program is that it does not scrape reply
        #
                
        for comment in wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#content-text"))):
            comment_data.append(comment.text)

        for author in wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#author-text > span"))):
            author_data.append(author.text)
                    
        table['comment'] = comment_data
                
        table['author'] = author_data
                
        table.drop_duplicates(subset='comment', keep='first', inplace=True)        
        
        print('detect '+str(Except)+' Exceptions')
    
    #######################################################
    #    output 2: a table to create in database later    #
    #######################################################
            
    table['collected_date'] = dt.date.today()  
    table['song_name'] = song_name
            
    leng = len(table)
    print('collected '+str(leng)+' comments for '+video_title)
    
    #
    #  making a new table in the database named by the video_title with data table
    #
    
    print('end processing for... '+video_title)
    print('--------------------------------------------------------\n')
    
    return(row, table)

In [15]:
def insert_record_to_video_list_and_create_comment_table_accordingly(row):
    
    #
    #    row = (video_title, song_name, group_id, r_date, v_address) 
    #
    
    #
    # pre-processing row
    #
       
    rr = list(row)
    
    try:
        dd = dt.datetime.strptime(rr[3], '%b %d, %Y').date()
        rr[3] = dd
    except:
        rr[3] = None

    row = tuple(rr)
    print(row)
    
    #
    # inserting data to song_list through sql
    #
        
    insert_song_list_query = """

    INSERT INTO video_list(video_title, song_name, group_id, released_date, youtube_add)
        VALUES ( %s, %s, %s, %s, %s)
        
        """
    
    try:
        with connect(host=h, user=u, password=pw, database='who_can_be_kpop') as connection:
        
            with connection.cursor() as cursor:            
                cursor.execute(insert_song_list_query, row)
                connection.commit()
                
                print('finished inserting one rwo into song_list')
                
    except Error as e:
        print(e)
    
    #
    # creating table through sql
    # and inserting data
    #
    
    create_table = """

    CREATE TABLE `%s`  (
        video_id INT,
        comment VARCHAR(1500),
        author VARCHAR(100),
        collected_date DATE,
        
        FOREIGN KEY(video_id) REFERENCES video_list(id)
        ) DEFAULT CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci
        
        """
    
    old_song_name = str(row[1])
    new_song_name = re.sub(' ', '_', old_song_name)
    table_name = 'comment_'+new_song_name
    
    try:
        with connect(host=h, user=u, password=pw,
                     database='who_can_be_kpop',
                     ) as connection:
        
            with connection.cursor() as cursor:            
                cursor.execute(create_table % (table_name) )
                connection.commit()
            print('finished creating table: '+ table_name)
            
    except Error as e:
        print(e)
        
    print('')

In [16]:
def find_video_id(song_name):
    
    #
    #  song_name: string, lowercase only, space allowed
    #
    
    paired_video_id = dict()

    select_paired_video_id = """ SELECT id, song_name FROM video_list """

    try:
        with connect(host=h, user=u, password=pw, database='who_can_be_kpop') as connection:
        
            with connection.cursor() as cursor:            
                cursor.execute(select_paired_video_id)
                for row in cursor.fetchall():
                
                    a, b = row
                    paired_video_id[b] = a
                
                
    except Error as e:
        print(e)
    
    return(paired_video_id[song_name])

In [17]:
def insert_comments(table):
    
    #
    #    table with columns(comment, author, collected_date, song_name)
    #
    
    counter = 0
    
    for i, r in table.iterrows():
        
        song_name = r['song_name']
        video_id = find_video_id(song_name)
        
        new_song_name = re.sub(' ', '_', song_name)
        table_name = 'comment_'+new_song_name
        
        comment = r['comment']
        author = r['author']
        collected_date = r['collected_date']
        
        record = (video_id, comment, author, collected_date)
    
        insert_comment_query = """

            INSERT INTO %s (video_id, comment, author, collected_date)
                VALUES ( %%s, %%s, %%s, %%s)
        
                            """
        
        insert_comment_query = insert_comment_query % table_name
        
        try:
            with connect(host=h, user=u, password=pw, database='who_can_be_kpop') as connection:
        
                with connection.cursor() as cursor:            
                    cursor.execute(insert_comment_query, record)
                    connection.commit()
                    
        except Error as e:
            print(e)
            
        counter += 1
        
        if counter % 200 == 0:
            print('finished inserting '+str(counter)+' comments')

In [18]:
start = dt.datetime.now()

## dict key doesn't matter

data_dict_list = [

    {'luv wrong': (find_group_id('exp edition'), 'luv wrong', "https://www.youtube.com/watch?v=u0VubbfytQs")},
    {'popsicle': (find_group_id('uhsn'), 'popsicle', "https://www.youtube.com/watch?v=g_dqiyxZO_Y")},
    {'your turn': (find_group_id('kaachi'), 'your turn', "https://www.youtube.com/watch?v=1NWzao4xf1E")},
    {'tonight': (find_group_id('blackswan'), 'tonight', "https://www.youtube.com/watch?v=Yb_K5-IIKgg")},
    {'breakout': (find_group_id('prisma'), 'breakout', "https://www.youtube.com/watch?v=17xRBNkMp14")},
    {'piri': (find_group_id('5 high'), 'piri', "https://www.youtube.com/watch?v=zI7XMeqyNA8")},
    
    {'q a': (find_group_id('cherry bullet'), 'q a', "https://www.youtube.com/watch?v=5LCGn9UFNAY")},
    {'flavor': (find_group_id('fanatics'), 'flavor', "https://www.youtube.com/watch?v=lSIx8K_NMsk")},
    {'woo ah': (find_group_id('woo!ah!'), 'woo ah', "https://www.youtube.com/watch?v=X2FB37r4Oyw")},

    ]

for ddd in data_dict_list:
    
    #
    # scrap comment from the youtube video provided
    # and return a row (for video_list) and a table (for comment_table)
    #
    
    row, table = scrapper(ddd, 1000)
    
    #
    # insert the row into table video_list
    # and create comment table accordingly
    #
    
    insert_record_to_video_list_and_create_comment_table_accordingly(row)
    
    #
    # insert comments from table into the comment_table
    #
    
    insert_comments(table)
    
    print('finished one video: ', ddd.keys())
    
end = dt.datetime.now()

print('program takes: ', end - start )

detect 2 Exceptions
collected 1017 comments for EXP EDITION (이엑스피 에디션) - LUV/WRONG (Official Music Video 뮤직비디오)
end processing for... EXP EDITION (이엑스피 에디션) - LUV/WRONG (Official Music Video 뮤직비디오)
--------------------------------------------------------

('EXP EDITION (이엑스피 에디션) - LUV/WRONG (Official Music Video 뮤직비디오)', 'luv wrong', 4, datetime.date(2015, 11, 6), 'https://www.youtube.com/watch?v=u0VubbfytQs')
finished inserting one rwo into song_list
finished creating table: comment_luv_wrong

1406 (22001): Data too long for column 'comment' at row 1
finished inserting 200 comments
1406 (22001): Data too long for column 'comment' at row 1
finished inserting 400 comments
finished inserting 600 comments
finished inserting 800 comments
finished inserting 1000 comments
finished one video:  dict_keys(['luv wrong'])
detect 2 Exceptions
collected 1014 comments for 유학소녀 (UHSN) - 팝시클 (POPSICLE) Music Video
end processing for... 유학소녀 (UHSN) - 팝시클 (POPSICLE) Music Video
------------------------

<h2> Check Whether Data were Stored Successfully with Select </h2>

In [20]:
sql = """ SELECT * FROM video_list"""

try:
    with connect(host=h, user=u, password=pw, database='who_can_be_kpop') as connection:
        
        with connection.cursor() as cursor:            
            cursor.execute(sql)
            
            for item in cursor.fetchall():
                
                print(item)
                
                
except Error as e:
    print(e)

(1, 'EXP EDITION (이엑스피 에디션) - LUV/WRONG (Official Music Video 뮤직비디오)', 'luv wrong', 4, datetime.date(2015, 11, 6), 'https://www.youtube.com/watch?v=u0VubbfytQs')
(2, '유학소녀 (UHSN) - 팝시클 (POPSICLE) Music Video', 'popsicle', 3, datetime.date(2019, 7, 11), 'https://www.youtube.com/watch?v=g_dqiyxZO_Y')
(3, "KAACHI - 'Your Turn' (Official Music Video)", 'your turn', 6, datetime.date(2020, 4, 28), 'https://www.youtube.com/watch?v=1NWzao4xf1E')
(4, 'MVㅣBLACKSWAN (블랙스완) - TonightㅣGoodbye RANIA', 'tonight', 1, datetime.date(2020, 10, 16), 'https://www.youtube.com/watch?v=Yb_K5-IIKgg')
(5, 'PRISMA - Breakout [Official Video]', 'breakout', 2, None, 'https://www.youtube.com/watch?v=17xRBNkMp14')
(6, "5 High – 'PIRI' (Dreamcatcher COVER)", 'piri', 7, datetime.date(2020, 5, 3), 'https://www.youtube.com/watch?v=zI7XMeqyNA8')
(7, '[MV] 체리블렛(Cherry Bullet) _ Q&A', 'q a', 8, datetime.date(2019, 1, 21), 'https://www.youtube.com/watch?v=5LCGn9UFNAY')
(8, '[MV] FANATICS-FLAVOR(파나틱스-플레이버) _ MILKSHAKE (Korea